In [ ]:
"""ProtT5-XL embedding extraction for food-derived protein sequences."""

'\nPLM Feature Extraction Script\n==============================\nGenerates embedding CSVs for: ESM2, ProtBERT, ProGen2, ProtGPT2, ProtT5, Ankh\n\nDesign guarantees:\n  - LEAKPROOF: embeddings are computed purely from sequence content,\n    no label information touches the model at any point.\n  - CHECKPOINT RECOVERY: each PLM writes a .done sentinel after saving.\n    Re-running skips completed PLMs automatically.\n  - KAGGLE-SAFE: GPU memory is released between PLMs via del + gc + cuda.empty_cache.\n  - REPRODUCIBLE: sequences are aligned to the label CSV by index; ordering\n    is fixed and documented.\n\nUsage (Kaggle):\n  Place your label CSV at INPUT_LABELS_PATH.\n  Set OUTPUT_DIR to /kaggle/working/.\n  Run the whole notebook cell — if it dies, re-run the same cell; it picks\n  up from the last completed PLM.\n\nOutputs (one per PLM):\n  features_esm2_<N>.csv\n  features_protbert_<N>.csv\n  features_progen2_<N>.csv\n  features_protgpt2_<N>.csv\n  features_prott5_<N>.csv\n  featu

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these for your Kaggle setup
# ──────────────────────────────────────────────────────────────────────────────
INPUT_LABELS_PATH = "/kaggle/input/datasets/harshiikkaa/food-6356/Food_6356.csv"
OUTPUT_DIR        = "/kaggle/working"
BATCH_SIZE        = 16      
MAX_SEQ_LEN       = 1024   
LABEL_COL_IDX     = 1       
SEQ_COL_IDX       = 0       
# ──────────────────────────────────────────────────────────────────────────────

import os, gc, sys, json, time, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: Tesla T4


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load labels and sequences 
# ──────────────────────────────────────────────────────────────────────────────

def load_dataset(path):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

    # Auto-detect sequence column: first column that looks like AA strings
    seq_col = None
    if SEQ_COL_IDX is not None:
        seq_col = df.columns[SEQ_COL_IDX]
    else:
        for col in df.columns:
            sample = str(df[col].iloc[0])
            if all(c in "ACDEFGHIKLMNPQRSTVWYXUBZOacdefghiklmnpqrstvwyx" for c in sample[:20]):
                seq_col = col
                break
    if seq_col is None:
        raise ValueError("Could not auto-detect sequence column. Set SEQ_COL_IDX manually.")

    label_col = df.columns[LABEL_COL_IDX]
    sequences = df[seq_col].astype(str).tolist()
    labels    = df[label_col].tolist()
    print(f"Sequence col: '{seq_col}'  |  Label col: '{label_col}'")
    print(f"Sequence length stats — min: {min(len(s) for s in sequences)}, "
          f"max: {max(len(s) for s in sequences)}, "
          f"mean: {np.mean([len(s) for s in sequences]):.0f}")
    return sequences, labels, df

sequences, labels, df_orig = load_dataset(INPUT_LABELS_PATH)
N = len(sequences)

# Save split_index.csv for traceability (row order anchor)
split_idx_path = os.path.join(OUTPUT_DIR, "split_index.csv")
if not os.path.exists(split_idx_path):
    pd.DataFrame({"global_index": range(N), "label": labels}).to_csv(split_idx_path, index=False)
    print(f"Saved: {split_idx_path}")

Loaded 6356 rows, columns: ['Sequence', 'Label']
Sequence col: 'Sequence'  |  Label col: 'Label'
Sequence length stats — min: 60, max: 1022, mean: 392
Saved: /kaggle/working/split_index.csv


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 2 — Checkpoint helpers
# ──────────────────────────────────────────────────────────────────────────────

def done_path(plm_name):
    return os.path.join(OUTPUT_DIR, f".{plm_name}.done")

def out_path(plm_name):
    return os.path.join(OUTPUT_DIR, f"features_{plm_name}_{N}.csv")

def is_done(plm_name):
    return os.path.exists(done_path(plm_name)) and os.path.exists(out_path(plm_name))

def mark_done(plm_name):
    with open(done_path(plm_name), "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))

def save_features(plm_name, embeddings: np.ndarray):
    """Save embeddings with dimension column names. No label column."""
    assert embeddings.shape[0] == N, \
        f"Row count mismatch: got {embeddings.shape[0]}, expected {N}"
    cols = [f"dim_{i}" for i in range(embeddings.shape[1])]
    pd.DataFrame(embeddings, columns=cols).to_csv(out_path(plm_name), index=False)
    mark_done(plm_name)
    print(f"  ✓ Saved {out_path(plm_name)}  shape={embeddings.shape}")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 3 — Batched embedding loop
# ──────────────────────────────────────────────────────────────────────────────

def embed_in_batches(encode_fn, sequences, batch_size=BATCH_SIZE, desc=""):
    """
    encode_fn(batch: List[str]) -> np.ndarray of shape (len(batch), D)
    Returns full (N, D) array.
    """
    all_embs = []
    n = len(sequences)
    for start in range(0, n, batch_size):
        batch = sequences[start : start + batch_size]
        emb   = encode_fn(batch)
        all_embs.append(emb)
        done  = min(start + batch_size, n)
        print(f"\r  {desc} {done}/{n}", end="", flush=True)
    print()
    return np.vstack(all_embs)





  ESM2 (esm2_t30_150M_UR50D)


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/595M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ESM2 6356/6356
  ✓ Saved /kaggle/working/features_esm2_6356.csv  shape=(6356, 640)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ProtT5-XL-UniRef50
# ──────────────────────────────────────────────────────────────────────────────

PLM_NAME = "prott5"

if is_done(PLM_NAME):
    print(f"[SKIP] {PLM_NAME} already done.")
else:
    print(f"\n{'='*60}\n  ProtT5-XL (Rostlab/prot_t5_xl_uniref50)\n{'='*60}")
    from transformers import T5Tokenizer, T5EncoderModel

    tok = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
    mdl = T5EncoderModel.from_pretrained(
        "Rostlab/prot_t5_xl_uniref50", torch_dtype=torch.float16   # fp16 to fit T4
    ).eval().to(DEVICE)

    def encode_prott5(batch):
        # ProtT5 expects space-separated AAs; replace uncommon AAs with X
        seqs = [" ".join(list(s[:MAX_SEQ_LEN].upper().replace("U","X").replace("Z","X").replace("O","X").replace("B","X")))
                for s in batch]
        enc  = tok(seqs, return_tensors="pt", padding=True,
                   truncation=True, max_length=MAX_SEQ_LEN + 1).to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        hidden = out.last_hidden_state.float()          # back to float32
        mask   = enc["attention_mask"].unsqueeze(-1).float()
        emb    = (hidden * mask).sum(1) / mask.sum(1)
        return emb.cpu().numpy()

    embs = embed_in_batches(encode_prott5, sequences, batch_size=4, desc="ProtT5")
    save_features(PLM_NAME, embs)

    del tok, mdl, embs
    free_gpu()



  ProtT5-XL (Rostlab/prot_t5_xl_uniref50)


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ProtT5 7170/7170
  ✓ Saved /kaggle/working/features_prott5_7170.csv  shape=(7170, 1024)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DONE
# ──────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print(" PLM COMPLETE")
print("="*60)

completed = []
for name in ["prott5"]:
    status = "✓" if is_done(name) else "✗ MISSING"
    path   = out_path(name)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if os.path.exists(path) else "—"
    print(f"  {status}  {name:<12}  {size}")
    if is_done(name):
        completed.append(name)

print(f"\nCompleted: {len(completed)}/1")
print(f"Output dir: {OUTPUT_DIR}")


  ALL PLMs COMPLETE
  ✓  esm2          55.1 MB
  ✓  protbert      90.4 MB
  ✗ MISSING  progen2       —
  ✓  protgpt2      111.4 MB
  ✓  prott5        90.7 MB
  ✓  ankh          69.9 MB

Completed: 5/6
Output dir: /kaggle/working
